# Heart Disease Prediction

This notebook implements a heart disease prediction workflow using Python, pandas, scikit-learn, and matplotlib. The model is trained on the UCI Cleveland Heart Disease dataset.

## 1. Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc

plt.style.use('ggplot')
%matplotlib inline

## 2. Loading dataset and previewing data

In [ ]:
df = pd.read_csv('heart_disease_data.csv')
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
print('\nValue counts for target:')
print(df['target'].value_counts())

## 3. Data preprocessing

In [ ]:
# Convert target to binary: 0 = no disease, 1 = disease
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# Replace missing values marked with '?' and convert to numeric
for col in ['ca', 'thal']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values using median for numeric columns
df['ca'].fillna(df['ca'].median(), inplace=True)
df['thal'].fillna(df['thal'].median(), inplace=True)

print('Missing values after preprocessing:')
print(df.isna().sum())
df.head()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Model building and training

In [ ]:
# Scikit-learn Logistic Regression
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Random Forest classifier for comparison
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Custom logistic regression from scratch
class ScratchLogisticRegression:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for _ in range(self.n_iterations):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigmoid(linear_model)
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_model)
    def predict(self, X):
        probabilities = self.predict_proba(X)
        return np.where(probabilities >= 0.5, 1, 0)

scratch_model = ScratchLogisticRegression(learning_rate=0.1, n_iterations=5000)
scratch_model.fit(X_train_scaled, y_train.values)


## 5. Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.predict_proba(X_test)
    print(f'--- {model_name} ---')
    print('Accuracy:', accuracy_score(y_test, y_pred))
    print('Precision:', precision_score(y_test, y_pred))
    print('Recall:', recall_score(y_test, y_pred))
    print('F1 score:', f1_score(y_test, y_pred))
    print('ROC AUC:', roc_auc_score(y_test, y_proba))
    print('Confusion matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('Classification report:')
    print(classification_report(y_test, y_pred))
    print()
    return y_pred, y_proba

lr_pred, lr_proba = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')
rf_pred, rf_proba = evaluate_model(rf_model, X_test, y_test, 'Random Forest')
scratch_pred = scratch_model.predict(X_test_scaled)
scratch_proba = scratch_model.predict_proba(X_test_scaled)
print('--- Scratch Logistic Regression ---')
print('Accuracy:', accuracy_score(y_test, scratch_pred))
print('Precision:', precision_score(y_test, scratch_pred))
print('Recall:', recall_score(y_test, scratch_pred))
print('F1 score:', f1_score(y_test, scratch_pred))
print('ROC AUC:', roc_auc_score(y_test, scratch_proba))
print('Confusion matrix:')
print(confusion_matrix(y_test, scratch_pred))
print('Classification report:')
print(classification_report(y_test, scratch_pred))


## 6. Visualization

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0, 1], ['No Disease', 'Disease'])
plt.yticks([0, 1], ['No Disease', 'Disease'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, lr_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Reflections on model performance

- The dataset was preprocessed by converting the target into a binary label and filling missing numerical values.
- Logistic Regression produced a reliable baseline with good generalization on the test set.
- Random Forest often improves performance with non-linear patterns, while the scratch logistic regression shows how the model works internally.
- The confusion matrix and ROC curve help identify classification strengths and areas for improvement.
- Future improvements can include feature engineering, cross-validation, and hyperparameter search.